# 🔬 Capítulo 4: Recursión — Diseño y Análisis

## Estructuras de Datos y Algoritmos · Lic. en Sistemas

| Campo | Valor |
|-------|-------|
| **Capítulo** | 4 — Recursión |
| **Notebook** | 3 / 3 — Diseño y Análisis |
| **Fuente** | Goodrich, Tamassia & Goldwasser, Cap. 4 |
| **Temas** | Tipos de recursión · Recurrencias · Memoización · Tail recursion |

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap04/03_Diseno_Analisis_Recursion_Teoria.ipynb)

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, el estudiante será capaz de:

1. **Clasificar** un algoritmo recursivo en lineal, binario, múltiple o de cola
2. **Plantear** la relación de recurrencia T(n) de un algoritmo recursivo y resolverla por sustitución o árbol
3. **Aplicar memoización** para eliminar cómputo duplicado en recursión con subproblemas solapados
4. **Convertir** una recursión de cola a forma iterativa equivalente
5. **Diseñar** soluciones recursivas con el patrón: caso base → hipótesis inductiva → caso general

---


## 📖 Contenidos

### 🌿 Sección 1 — Tipos de Recursión
1. Recursión lineal (un solo caso recursivo)
2. Recursión binaria (dos casos recursivos)
3. Recursión múltiple (más de dos casos)
4. Recursión de cola (tail recursion)

### ⚡ Sección 2 — Análisis de Complejidad Recursiva
1. Relaciones de recurrencia: T(n) = aT(n/b) + f(n)
2. Casos comunes: O(n), O(log n), O(n log n), O(2^n)
3. Árbol de recursión como herramienta visual

### 🧠 Sección 3 — Diseño de Algoritmos Recursivos
1. Patrón: definir el caso base primero
2. Confiar en la hipótesis inductiva
3. Comparación recursión vs iteración
4. Cómo eliminar tail recursion

---


## 🔧 Configuración del Entorno

Importamos las librerías necesarias para este capítulo:

In [ ]:
# Librerías estándar de Python
import sys
import time
import math
from typing import List, Optional, Any, Union

# Librerías para visualización y análisis
import matplotlib.pyplot as plt
import numpy as np

# Configuración de matplotlib
plt.style.use('default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print(f"Python version: {sys.version}")
print(f"Entorno configurado correctamente ✅")

---

# 🌿 Sección 1: Tipos de Recursión

## ¿Por qué clasificar?

Conocer el *tipo* de recursión que tiene un algoritmo ayuda a:
- Predecir su **complejidad** sin hacer la cuenta a mano
- Detectar si se pueden aplicar optimizaciones (memoización, tail-call optimization, etc.)
- Comunicar el diseño con precisión

## Cuatro tipos canónicos

| Tipo | Descripción | Llamadas recursivas | Ejemplo clásico | Complejidad típica |
|------|-------------|--------------------|-----------------|--------------------|
| **Lineal** | Una sola llamada recursiva por caso | 1 | `factorial`, `reverse_list` | O(n) |
| **Binaria** | Dos llamadas recursivas por caso | 2 | Fibonacci, `merge_sort` | O(2^n) ó O(n log n) |
| **Múltiple** | Más de dos llamadas | 3+ | Permutaciones, backtracking | O(k^n) |
| **De cola** | La llamada recursiva es la *última* operación | 1 | `factorial_tail`, `sum_tail` | O(n) → optimizable a O(1) espacio |

> **Nota**: la recursión de cola no es exponencial, pero su *estructura* es diferente porque no hay trabajo pendiente al retornar.


In [ ]:
from typing import List
import sys
sys.setrecursionlimit(5000)

# ── 1. Recursión LINEAL ──────────────────────────────────────────────
# Una llamada recursiva; el trabajo se hace antes o después, no ambas.

def reverse_list(lst: List, lo: int = 0, hi: int = None) -> None:
    """Invierte lst en el lugar. Recursión lineal O(n)."""
    if hi is None:
        hi = len(lst) - 1
    if lo >= hi:
        return                          # caso base
    lst[lo], lst[hi] = lst[hi], lst[lo]
    reverse_list(lst, lo + 1, hi - 1)  # única llamada recursiva

a = [1, 2, 3, 4, 5]
reverse_list(a)
assert a == [5, 4, 3, 2, 1]
print("✓ reverse_list lineal:", a)


# ── 2. Recursión BINARIA ─────────────────────────────────────────────
# Dos llamadas recursivas por caso → árbol de llamadas de altura n.

def suma_binaria(lst: List[int], lo: int, hi: int) -> int:
    """Suma por divide y vencerás. Recursión binaria O(n)."""
    if lo == hi:
        return lst[lo]                          # caso base
    mid = (lo + hi) // 2
    return suma_binaria(lst, lo, mid) + suma_binaria(lst, mid+1, hi)  # 2 llamadas

data = list(range(1, 9))   # [1..8]
resultado = suma_binaria(data, 0, len(data)-1)
assert resultado == sum(data)
print(f"✓ suma_binaria: {resultado} == {sum(data)}")


# ── 3. Recursión MÚLTIPLE ────────────────────────────────────────────
# Más de dos llamadas; genera árboles de búsqueda, permutaciones, etc.

def permutaciones(lst: List) -> List[List]:
    """Genera todas las permutaciones. Recursión múltiple O(n!)."""
    if len(lst) <= 1:
        return [lst[:]]
    result = []
    for i in range(len(lst)):
        lst[0], lst[i] = lst[i], lst[0]
        for perm in permutaciones(lst[1:]):    # llamada recursiva por cada i
            result.append([lst[0]] + perm)
        lst[0], lst[i] = lst[i], lst[0]
    return result

perms = permutaciones([1, 2, 3])
assert len(perms) == 6                   # 3! = 6
print(f"✓ permutaciones([1,2,3]): {len(perms)} permutaciones")


# ── 4. Recursión DE COLA ─────────────────────────────────────────────
# La llamada recursiva es la última instrucción → el compilador/intérprete
# puede optimizarla (Python NO lo hace automáticamente, pero el patrón importa).

def factorial_tail(n: int, acum: int = 1) -> int:
    """Factorial con acumulador. Tail recursion O(n)."""
    if n == 0:
        return acum                           # caso base
    return factorial_tail(n - 1, n * acum)  # tail call: última operación

assert factorial_tail(5) == 120
assert factorial_tail(0) == 1
print(f"✓ factorial_tail(5) = {factorial_tail(5)}")
print(f"✓ factorial_tail(0) = {factorial_tail(0)}")


### 🔍 Observaciones clave

- `reverse_list`: **lineal** — el trabajo (swap) está *fuera* de la llamada recursiva; hay pending work al retornar
- `suma_binaria`: **binaria** — divide el rango; igual número de nodos en el árbol que elementos (O(n) total)
- `permutaciones`: **múltiple** — exactamente n! hojas en el árbol → O(n · n!) tiempo
- `factorial_tail`: **de cola** — el acumulador `acum` lleva todo el estado; no hay trabajo pendiente al retornar

**¿Por qué importa la recursión de cola?**  
En lenguajes con *TCO* (Tail-Call Optimization) — Haskell, Scheme, Kotlin — se convierte automáticamente en un loop y usa **O(1) espacio de pila**. Python *no* hace TCO, pero el patrón es importante al portear código o explicar complejidad espacial.

---


---

# ⚡ Sección 2: Relaciones de Recurrencia y Análisis de Complejidad

## ¿Qué es una relación de recurrencia?

Una **relación de recurrencia** expresa el costo de un algoritmo recursivo en función de su propio costo para entradas más pequeñas.

$$T(n) = a \cdot T\!\left(\frac{n}{b}\right) + f(n)$$

Donde:
- $a$ = número de llamadas recursivas
- $n/b$ = tamaño del subproblema
- $f(n)$ = trabajo realizado *fuera* de las llamadas recursivas

## Casos típicos que debes memorizar

| Recurrencia | Solución | Ejemplo |
|-------------|----------|---------|
| $T(n) = T(n-1) + O(1)$ | $O(n)$ | `factorial`, `reverse_list` |
| $T(n) = T(n-1) + O(n)$ | $O(n^2)$ | Insertion sort naive |
| $T(n) = T(n/2) + O(1)$ | $O(\log n)$ | Búsqueda binaria |
| $T(n) = 2T(n/2) + O(n)$ | $O(n \log n)$ | Merge sort |
| $T(n) = 2T(n-1) + O(1)$ | $O(2^n)$ | Fibonacci naïve, Hanói |
| $T(n) = T(n-1) + T(n-2) + O(1)$ | $O(2^n)$ | Fibonacci (análisis preciso) |

## Método del árbol de recursión

El árbol de recursión visualiza cómo se *ramifica* el cómputo:
- Cada nivel contribuye $f(n/b^k)$ trabajo
- La profundidad máxima es $\log_b n$
- Sumar el trabajo por niveles da la complejidad total


In [ ]:
import sys, functools
sys.setrecursionlimit(10_000)

# Instrumentamos varias funciones para contar llamadas y verificar la recurrencia

def make_counter():
    return [0]

# ── T(n) = T(n-1) + O(1)  →  O(n) ──────────────────────────────────
counter_lin = make_counter()
def fact_count(n):
    counter_lin[0] += 1
    if n <= 1: return 1
    return n * fact_count(n - 1)

# ── T(n) = T(n/2) + O(1)  →  O(log n) ──────────────────────────────
counter_log = make_counter()
def power_fast_count(x, n):
    counter_log[0] += 1
    if n == 0: return 1
    half = power_fast_count(x, n // 2)
    return half * half if n % 2 == 0 else half * half * x

# ── T(n) = 2T(n-1) + O(1)  →  O(2^n) ───────────────────────────────
counter_exp = make_counter()
def fib_count(n):
    counter_exp[0] += 1
    if n <= 1: return n
    return fib_count(n-1) + fib_count(n-2)

# Medir para n = 1..20
import math
ns = list(range(1, 21))
calls_lin, calls_log, calls_exp = [], [], []

for n in ns:
    counter_lin[0] = 0;  fact_count(n);         calls_lin.append(counter_lin[0])
    counter_log[0] = 0;  power_fast_count(2, n); calls_log.append(counter_log[0])
    counter_exp[0] = 0;  fib_count(n);           calls_exp.append(counter_exp[0])

# Imprimir tabla
print(f"{'n':>4} | {'T(n-1)+O(1) [O(n)]':>18} | {'T(n/2)+O(1) [O(log n)]':>22} | {'2T(n-1)+O(1) [O(2^n)]':>22}")
print("-" * 75)
for i, n in enumerate(ns):
    print(f"{n:>4} | {calls_lin[i]:>18} | {calls_log[i]:>22} | {calls_exp[i]:>22}")


### 📈 Visualización: confirmando las recurrencias empíricamente


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ns_arr    = np.array(ns)
lin_arr   = np.array(calls_lin)
log_arr   = np.array(calls_log)
exp_arr   = np.array(calls_exp, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Escala lineal ──────────────────────────────────────────────────
ax = axes[0]
ax.plot(ns_arr, lin_arr, 'o-', label='O(n) — T(n-1)+O(1)',   color='steelblue', lw=2)
ax.plot(ns_arr, log_arr, 's-', label='O(log n) — T(n/2)+O(1)', color='green', lw=2)
ax.plot(ns_arr, exp_arr, '^-', label='O(2ⁿ) — 2T(n-1)+O(1)', color='crimson', lw=2)
ax.set_title("Llamadas recursivas (escala lineal)", fontsize=13)
ax.set_xlabel("n"); ax.set_ylabel("Llamadas")
ax.legend(); ax.grid(True, alpha=0.4)

# ── Escala log ────────────────────────────────────────────────────
ax = axes[1]
ax.semilogy(ns_arr, lin_arr, 'o-', label='O(n)',     color='steelblue', lw=2)
ax.semilogy(ns_arr, log_arr, 's-', label='O(log n)', color='green', lw=2)
ax.semilogy(ns_arr, exp_arr, '^-', label='O(2ⁿ)',    color='crimson', lw=2)
ax.set_title("Mismos datos (escala logarítmica)", fontsize=13)
ax.set_xlabel("n"); ax.set_ylabel("log(Llamadas)")
ax.legend(); ax.grid(True, which='both', alpha=0.4)

plt.suptitle("Comparación empírica de recurrencias comunes", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import functools, time, sys
sys.setrecursionlimit(10_000)

# ── Memoización manual vs @lru_cache ─────────────────────────────────
# Muestra cómo implementar la caché a mano (para comprender @lru_cache)

def fib_memo_manual(n: int, memo: dict = None) -> int:
    """Fibonacci con caché manual."""
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fib_memo_manual(n - 1, memo) + fib_memo_manual(n - 2, memo)
    return memo[n]


@functools.lru_cache(maxsize=None)
def fib_lru(n: int) -> int:
    """Fibonacci con @lru_cache de Python."""
    if n <= 1:
        return n
    return fib_lru(n - 1) + fib_lru(n - 2)


# Verificación de equivalencia
for i in range(35):
    assert fib_memo_manual(i) == fib_lru(i), f"Mismatch en fib({i})"

print("✓ fib_memo_manual == fib_lru para n=0..34")
print(f"  fib(34) = {fib_lru(34)}")

# Diferencia de rendimiento respecto a versión naïve
print("\nTiempos para fib(30):")
t0 = time.perf_counter(); fib_memo_manual(30); print(f"  Manual memo: {time.perf_counter()-t0:.6f}s")
fib_lru.cache_clear()
t0 = time.perf_counter(); fib_lru(30);         print(f"  @lru_cache:  {time.perf_counter()-t0:.6f}s")


---

# 🧠 Sección 3: Diseño de Algoritmos Recursivos

## El patrón de los 3 pasos

Todo algoritmo recursivo sigue este esquema:

```
1. CASO BASE  →  solución directa para el caso trivial (n=0, lista vacía, etc.)
2. HIPÓTESIS  →  suponer que la función ya funciona correctamente para n-1
3. RECURSIÓN  →  usar el resultado de n-1 para construir la solución de n
```

> **Regla de oro**: Confía en la hipótesis inductiva. No traces mentalmente la recursión completa — si el caso base es correcto y el paso recursivo transforma el problema correctamente, la función es correcta para todo n.

## Cómo convertir tail recursion a iteración

La recursión de cola es **mecánicamente convertible** a un bucle:

```python
# Versión recursiva de cola
def fact_tail(n, acum=1):
    if n == 0: return acum
    return fact_tail(n-1, n*acum)

# Versión iterativa equivalente
def fact_iter(n):
    acum = 1
    while n > 0:
        acum *= n
        n -= 1
    return acum
```

El acumulador en la recursión de cola se convierte en la **variable de estado del bucle**.


In [ ]:
from typing import List
import sys
sys.setrecursionlimit(5000)

# ══════════════════════════════════════════════════════════════════════
# Ejemplo 1: Diseño top-down — suma de lista
# Estrategia: ¿cómo sumo una lista de n elementos si ya sé sumar n-1?
# Respuesta: suma(lst) = lst[0] + suma(lst[1:])
# ══════════════════════════════════════════════════════════════════════

def suma_lista(lst: List[int]) -> int:
    """Suma todos los elementos de lst. Recursión lineal O(n)."""
    if not lst:          # caso base: lista vacía
        return 0
    return lst[0] + suma_lista(lst[1:])  # hipótesis: suma_lista(lst[1:]) ya funciona

assert suma_lista([]) == 0
assert suma_lista([1]) == 1
assert suma_lista([1, 2, 3, 4, 5]) == 15
print("✓ suma_lista:", [suma_lista(list(range(k))) for k in range(6)])


# ══════════════════════════════════════════════════════════════════════
# Ejemplo 2: Conversión tail recursion → iterativo
# ══════════════════════════════════════════════════════════════════════

def fact_tail(n: int, acum: int = 1) -> int:
    """Factorial de cola."""
    if n <= 0: return acum
    return fact_tail(n - 1, n * acum)

def fact_iter(n: int) -> int:
    """Factorial iterativo — equivalente mecánico de fact_tail."""
    acum = 1
    while n > 0:
        acum *= n
        n -= 1
    return acum

for i in range(10):
    assert fact_tail(i) == fact_iter(i), f"Fallo en factorial({i})"
print("✓ fact_tail == fact_iter para n=0..9")


# ══════════════════════════════════════════════════════════════════════
# Ejemplo 3: ¿Cuándo NO usar recursión?
# Cuando la profundidad es > ~1000 y no hay TCO, preferir iteración.
# ══════════════════════════════════════════════════════════════════════

def suma_iterativa(n: int) -> int:
    """Suma 1+2+...+n iterativa. Segura para n grande."""
    return n * (n + 1) // 2     # O(1) fórmula de Gauss

def suma_recursiva(n: int) -> int:
    """Suma 1+2+...+n recursiva. Insegura para n > ~900 en Python."""
    if n <= 0: return 0
    return n + suma_recursiva(n - 1)

for n in [0, 1, 10, 100]:
    assert suma_iterativa(n) == suma_recursiva(n) == n*(n+1)//2

print("✓ suma_iterativa == suma_recursiva para n pequeños")
print(f"  Para n=1000, usa la fórmula: {suma_iterativa(1000)}")


### 🔄 Eliminación de Tail Recursion: Convertidor Automático

Demostramos el patrón mecánico para convertir *cualquier* recursión de cola a iteración:


In [ ]:
from typing import List

# Patrón: recursión de cola con acumulador → while loop
# Regla:   parámetros acumulados → variables locales, llamada recursiva → update + continue

# ── Suma con acumulador ───────────────────────────────────────────────
def sum_tail(lst: List[int], i: int = 0, acum: int = 0) -> int:
    """Recursión de cola."""
    if i == len(lst): return acum
    return sum_tail(lst, i + 1, acum + lst[i])

def sum_iter(lst: List[int]) -> int:
    """Iterativa equivalente (manual TCO)."""
    i, acum = 0, 0
    while i < len(lst):
        i, acum = i + 1, acum + lst[i]
    return acum

# ── Búsqueda lineal con acumulador ───────────────────────────────────
def linear_search_tail(lst: List, target, i: int = 0) -> int:
    """Devuelve el índice del target, o -1. Recursión de cola."""
    if i >= len(lst):    return -1
    if lst[i] == target: return i
    return linear_search_tail(lst, target, i + 1)

def linear_search_iter(lst: List, target) -> int:
    """Equivalente iterativa."""
    for i, v in enumerate(lst):
        if v == target: return i
    return -1

# Verificaciones
data = [3, 1, 4, 1, 5, 9, 2, 6]
assert sum_tail(data) == sum_iter(data) == sum(data)
assert linear_search_tail(data, 9) == linear_search_iter(data, 9) == 5
assert linear_search_tail(data, 7) == linear_search_iter(data, 7) == -1
print("✓ sum_tail == sum_iter")
print("✓ linear_search_tail == linear_search_iter")


In [ ]:
import sys, time
sys.setrecursionlimit(5000)

# Comparación de uso de pila: recursión estándar vs. recursión de cola (simulada)
# Python no hace TCO pero podemos medir la profundidad de pila usada.

import traceback, io

def _depth_from_tb() -> int:
    """Utilidad: profundidad de la pila en el momento de la llamada."""
    return len(traceback.extract_stack())

profundidades_std  = {}
profundidades_tail = {}

def fact_std(n):
    """Factorial estándar — apila n marcos."""
    profundidades_std[n] = _depth_from_tb()
    if n <= 1: return 1
    return n * fact_std(n - 1)

def fact_tail2(n, acum=1):
    """Factorial de cola — también apila n marcos en Python (sin TCO)."""
    profundidades_tail[n] = _depth_from_tb()
    if n <= 0: return acum
    return fact_tail2(n - 1, n * acum)

for k in range(1, 11):
    fact_std(k)
    profundidades_tail.clear()
    fact_tail2(k)

print(f"{'n':>4} | {'fact_std profundidad':>22} | {'fact_tail profundidad':>22}")
print("-" * 55)
for k in range(1, 11):
    print(f"{k:>4} | {profundidades_std.get(k, '-'):>22} | {profundidades_tail.get(k, '-'):>22}")

print("\n⚠️  Python apila marcos igual en ambos casos (sin TCO)")
print("   En Haskell/Kotlin la versión de cola usaría profundidad constante.")


In [ ]:
import unittest

class TestDisenioRecursion(unittest.TestCase):

    # ── Tipos de recursión ────────────────────────────────────────────
    def test_reverse_list_lineal(self):
        a = [1, 2, 3, 4, 5]
        reverse_list(a)
        self.assertEqual(a, [5, 4, 3, 2, 1])

    def test_reverse_list_vacia(self):
        a = []
        reverse_list(a)
        self.assertEqual(a, [])

    def test_suma_binaria(self):
        data = list(range(1, 9))
        self.assertEqual(suma_binaria(data, 0, len(data)-1), sum(data))

    def test_factorial_tail(self):
        for n in range(8):
            self.assertEqual(factorial_tail(n), fact_tail(n))

    # ── Memoización ───────────────────────────────────────────────────
    def test_fib_memo_manual_conocidos(self):
        esperados = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
        for i, v in enumerate(esperados):
            self.assertEqual(fib_memo_manual(i), v)

    # ── Diseño top-down ───────────────────────────────────────────────
    def test_suma_lista(self):
        self.assertEqual(suma_lista([1, 2, 3, 4, 5]), 15)
        self.assertEqual(suma_lista([]),               0)

    def test_sum_tail_equals_builtin(self):
        data = [3, 1, 4, 1, 5, 9]
        self.assertEqual(sum_tail(data), sum(data))
        self.assertEqual(sum_iter(data), sum(data))


loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestDisenioRecursion)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print("\n✓ Suite completa –", result.testsRun, "tests,", len(result.failures), "fallos")


---

## 📊 Resumen del Capítulo 4: Recursión

### Tipos de recursión y sus propiedades

| Tipo | Llamadas | Complejidad típica | ¿Optimizable? | Ejemplo |
|------|----------|--------------------|---------------|---------|
| **Lineal** | 1 | O(n) | Solo si es de cola | `factorial`, `reverse_list` |
| **Binaria** | 2 | O(n log n) o O(2^n) | Con memo si hay solapamiento | `merge_sort`, Fibonacci |
| **Múltiple** | 3+ | O(k^n) | Con poda / memo | Permutaciones, backtracking |
| **De cola** | 1 (última op.) | O(n) tiempo, O(n) pila en Python | → Iterativa trivialmente | `factorial_tail`, `sum_tail` |

### Relaciones de recurrencia clave

| Recurrencia | Solución | Cómo resolverla |
|-------------|----------|-----------------|
| T(n) = T(n−1) + O(1) | O(n) | Suma telescópica |
| T(n) = T(n/2) + O(1) | O(log n) | Suma geométrica |
| T(n) = 2T(n/2) + O(n) | O(n log n) | Árbol: n trabajo por nivel × log n niveles |
| T(n) = 2T(n−1) + O(1) | O(2^n) | Árbol: 2^n hojas |

### Decisiones de diseño

```
¿Tiene subproblemas solapados?  →  Considera memoización / DP
¿La recursión es de cola?       →  Considera convertir a iterativa
¿n puede ser muy grande (>900)? →  Cuidado con stack overflow en Python
¿El output es de tamaño O(2^n)? →  La complejidad exponencial es inevitable
```

### Recursos del capítulo

- **Libro**: Goodrich, Tamassia & Goldwasser, Cap. 4 — pp. 148–195
- **Sec. 4.1**: Ilustraciones de recursión
- **Sec. 4.2**: Análisis de recursión
- **Sec. 4.3**: Recursión vs. iteración y tail recursion (pp. 179–183)
- **Apéndice B**: Notación matemática y resolución de recurrencias


---

## ✏️ Autoevaluación

1. **¿Cuál es la diferencia fundamental entre recursión binaria y recursión de cola?**  
   *Pista: número de llamadas y si hay trabajo pendiente al retornar.*

2. **Escribe la relación de recurrencia T(n) para `merge_sort`.**  
   *Pista: divide en dos mitades + merge que cuesta O(n).*

3. **¿En qué situación conviene usar memoización vs. programación dinámica bottom-up?**  
   *Pista: explorabilidad parcial del espacio de subproblemas.*

4. **Si una función tiene `sys.setrecursionlimit(1000)` y la llama con n=2000, ¿qué ocurre?**  
   *Respuesta esperada: `RecursionError: maximum recursion depth exceeded`*

5. **Convierte esta recursión de cola a iterativa:**

```python
def cuenta_atras(n, acum=None):
    if acum is None: acum = []
    if n < 0: return acum
    return cuenta_atras(n - 1, acum + [n])

# Tu versión iterativa:
def cuenta_atras_iter(n):
    ...  # implementa aquí

# Test: cuenta_atras(5) == [5, 4, 3, 2, 1, 0]
```


---

<div align="center">

**Capítulo 4 — Recursión | Notebook 3 de 3**  
*Data Structures & Algorithms in Python — Goodrich, Tamassia & Goldwasser*  
Facultad · Estructuras de Datos

[← Notebook 2: Ejemplos](02_Ejemplos_Recursion_Teoria.ipynb) &nbsp;|&nbsp; [Ejercicios Clase 1 →](Ejercicios_Clase1_Fundamentos.ipynb)

</div>
